# E4: Iterative Code Generation with CVX Debug Memory

CVX as **prefrontal cortex**: a generate-test-debug agent that recalls how similar errors were fixed.

### The Loop

```
1. Generate code (attempt N)
2. Run tests → pass? → done ✓
3. embed(error + code context) → CVX.search() for similar past errors
4. Extract continuation: "this error was fixed by..."
5. LLM(code + error + fix suggestions) → attempt N+1
6. → goto 2 (max 3 retries)
```

### Phase 1: Build Debug Trace Corpus

Generate imperfect MBPP solutions (T=0.8), capture errors, store traces as CVX episodes:
- Step 0: problem description
- Step 1: failed code attempt
- Step 2: error message
- Step 3: working solution (ground truth)

### Phase 2: Evaluate on HumanEval

| Condition | Description |
|-----------|-------------|
| **SinglePass** | One attempt, no retry |
| **Retry-NoMemory** | Retry with error feedback, no CVX |
| **Retry-CVX-Causal** | Retry + CVX error matching → fix continuations |

In [1]:
import os, json, time, traceback, re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from scipy import stats

USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
MAX_RETRIES = 3
TOP_K = 3

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'LLM: {LLM_MODEL}, Embedding: D={D}, Max retries: {MAX_RETRIES}')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7506.62it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM: qwen2.5-coder:7b-instruct, Embedding: D=384, Max retries: 3


In [2]:
from datasets import load_dataset

mbpp = load_dataset('google-research-datasets/mbpp', 'sanitized', split='train+test+prompt')
humaneval = load_dataset('openai/openai_humaneval', split='test')
print(f'MBPP: {len(mbpp)} problems, HumanEval: {len(humaneval)} problems')

MBPP: 384 problems, HumanEval: 164 problems


## 1. Build Debug Trace Corpus

For each MBPP problem:
1. Generate an imperfect solution (T=0.8)
2. Run the MBPP tests
3. If it fails: capture the error → store the trace
4. Episode = [problem, bad_code, error, good_code(ground truth)]

In [3]:
TRACES_PATH = str(DATA_DIR / 'debug_traces.json')

def generate_mbpp_solution(prompt, temperature=0.8):
    """Generate a (potentially imperfect) solution."""
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': 'Write a Python function. Return ONLY the code, no explanation, no markdown fences.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        max_tokens=512,
    )
    code = response.choices[0].message.content.strip()
    if code.startswith('```'):
        code = code.split('\n', 1)[1] if '\n' in code else code[3:]
    if code.endswith('```'):
        code = code[:-3]
    return code.strip()


def run_mbpp_tests(code, test_cases):
    """Run MBPP test assertions. Returns (passed, error_msg)."""
    test_code = code + '\n' + '\n'.join(test_cases)
    try:
        exec(test_code, {})
        return True, ''
    except Exception as e:
        return False, f'{type(e).__name__}: {str(e)[:200]}'


if os.path.exists(TRACES_PATH):
    with open(TRACES_PATH) as f:
        debug_traces = json.load(f)
    print(f'Loaded {len(debug_traces)} cached debug traces')
else:
    print('Building debug trace corpus from MBPP...')
    debug_traces = []
    n_failed = 0
    t0 = time.perf_counter()
    
    for idx, problem in enumerate(mbpp):
        prompt = problem['prompt']
        good_code = problem['code']
        test_cases = problem['test_list']
        
        try:
            bad_code = generate_mbpp_solution(prompt, temperature=0.8)
            passed, error_msg = run_mbpp_tests(bad_code, test_cases)
        except Exception as e:
            # LLM timeout or other error — skip
            continue
        
        if not passed and error_msg:
            debug_traces.append({
                'problem_idx': idx,
                'prompt': prompt,
                'bad_code': bad_code,
                'error': error_msg,
                'good_code': good_code,
                'test_cases': test_cases,
            })
            n_failed += 1
        
        if (idx + 1) % 50 == 0:
            elapsed = time.perf_counter() - t0
            eta = elapsed / (idx + 1) * (len(mbpp) - idx - 1)
            print(f'  [{idx+1}/{len(mbpp)}] {n_failed} failures ({elapsed:.0f}s, ETA {eta:.0f}s)')
            # Save incrementally
            with open(TRACES_PATH, 'w') as f:
                json.dump(debug_traces, f)
    
    with open(TRACES_PATH, 'w') as f:
        json.dump(debug_traces, f)
    print(f'Built {len(debug_traces)} debug traces ({len(debug_traces)}/{len(mbpp)} = {len(debug_traces)/len(mbpp):.1%} failure rate)')

# Analyze error types
error_types = {}
for t in debug_traces:
    etype = t['error'].split(':')[0]
    error_types[etype] = error_types.get(etype, 0) + 1

print(f'\n{len(debug_traces)} debug traces')
print('Error types:')
for etype, count in sorted(error_types.items(), key=lambda x: -x[1])[:10]:
    print(f'  {etype}: {count} ({count/len(debug_traces):.1%})')

Building debug trace corpus from MBPP...


  [50/384] 47 failures (23s, ETA 151s)


[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22, 24, 33, 36, 44, 48, 55, 66, 77, 88, 99]


  [100/384] 94 failures (48s, ETA 137s)


  [150/384] 139 failures (66s, ETA 102s)


...

5


  [300/384] 274 failures (127s, ETA 35s)


-2


  [350/384] 321 failures (146s, ETA 14s)


Built 353 debug traces (353/384 = 91.9% failure rate)

353 debug traces
Error types:
  NameError: 335 (94.9%)
  AssertionError: 12 (3.4%)
  TypeError: 4 (1.1%)
  ZeroDivisionError: 1 (0.3%)
  SyntaxError: 1 (0.3%)


## 2. Index Debug Traces in CVX

Each trace becomes a 4-step episode:
- Step 0: problem description (timestamp = idx * 1000)
- Step 1: failed code attempt (timestamp = idx * 1000 + 1)
- Step 2: error message (timestamp = idx * 1000 + 2)
- Step 3: working solution (timestamp = idx * 1000 + 3)

In [4]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'debug_traces.cvx')

if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded debug index: {len(index)} points')
else:
    print('Building CVX debug trace index...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    for ep_idx, trace in enumerate(debug_traces):
        # Step 0: problem
        v0 = embedder.encode(trace['prompt']).tolist()
        index.insert(entity_id=(ep_idx << 16) | 0, timestamp=ep_idx * 1000, vector=v0)
        
        # Step 1: failed code
        v1 = embedder.encode(trace['bad_code'][:500]).tolist()
        index.insert(entity_id=(ep_idx << 16) | 1, timestamp=ep_idx * 1000 + 1, vector=v1)
        
        # Step 2: error message (THE KEY EMBEDDING — this is what we search on)
        error_context = f"Error: {trace['error']}\nCode: {trace['bad_code'][:200]}"
        v2 = embedder.encode(error_context[:500]).tolist()
        index.insert(entity_id=(ep_idx << 16) | 2, timestamp=ep_idx * 1000 + 2, vector=v2)
        
        # Step 3: working solution
        v3 = embedder.encode(trace['good_code'][:500]).tolist()
        index.insert(entity_id=(ep_idx << 16) | 3, timestamp=ep_idx * 1000 + 3, vector=v3)
    
    index.save(INDEX_PATH)
    print(f'Built: {len(index)} points ({len(debug_traces)} episodes × 4 steps)')

print(f'Debug memory: {len(index)} vectors from {len(debug_traces)} error traces')

Building CVX debug trace index...


Built: 1412 points (353 episodes × 4 steps)
Debug memory: 1412 vectors from 353 error traces


## 3. Retrieval: Error-Matching with Fix Continuation

When the agent hits an error, it embeds `(error + code_context)` and searches CVX.
Matches on step 2 (error) return the fix (step 3). Matches on step 1 (bad code) return error + fix.

In [5]:
def retrieve_fix_suggestions(error_msg, code_context, top_k=TOP_K):
    """Search CVX for similar past errors and return how they were fixed.
    
    Query = embed(error + code context) — searches across ALL steps.
    When matching on step 2 (error), returns the fix (step 3).
    When matching on step 1 (bad code), returns error + fix.
    """
    query_text = f'Error: {error_msg}\nCode: {code_context[:300]}'
    query_vec = embedder.encode(query_text[:500]).tolist()
    
    results = index.search(vector=query_vec, k=top_k * 15)
    
    seen = set()
    suggestions = []
    for node_id, ts, score in results:
        ep_idx = ts // 1000
        match_step = ts % 1000
        
        if ep_idx in seen or ep_idx >= len(debug_traces):
            continue
        seen.add(ep_idx)
        
        trace = debug_traces[ep_idx]
        
        # Build the fix suggestion based on match step
        if match_step <= 2:
            # Matched on problem/code/error → return the fix
            suggestion = {
                'match_step': match_step,
                'similarity': score,
                'original_problem': trace['prompt'][:100],
                'original_error': trace['error'],
                'fix_code': trace['good_code'],
            }
        else:
            # Matched on solution → show it as reference
            suggestion = {
                'match_step': match_step,
                'similarity': score,
                'original_problem': trace['prompt'][:100],
                'original_error': '',
                'fix_code': trace['good_code'],
            }
        
        suggestions.append(suggestion)
        if len(suggestions) >= top_k:
            break
    
    return suggestions


def format_fix_context(suggestions):
    """Format fix suggestions for the LLM retry prompt."""
    if not suggestions:
        return ''
    
    lines = ['Here are fixes from similar past errors:\n']
    for i, s in enumerate(suggestions, 1):
        lines.append(f'### Fix {i} (from: {s["original_problem"][:60]}...)')
        if s['original_error']:
            lines.append(f'Similar error was: {s["original_error"][:100]}')
        lines.append(f'Working solution:')
        lines.append(f'```python')
        lines.append(s['fix_code'])
        lines.append(f'```')
        lines.append('')
    return '\n'.join(lines)


# Test: simulate an error and retrieve fixes
test_error = 'TypeError: unsupported operand type(s) for +: int and str'
test_code = 'def foo(x): return x + "hello"'
suggestions = retrieve_fix_suggestions(test_error, test_code)
print(f'Query error: {test_error}')
print(f'\nRetrieved {len(suggestions)} fix suggestions:')
for s in suggestions:
    print(f'  step={s["match_step"]}, sim={s["similarity"]:.3f}: {s["original_error"][:60]}')

Query error: TypeError: unsupported operand type(s) for +: int and str

Retrieved 3 fix suggestions:
  step=2, sim=0.865: NameError: name 'dog_age' is not defined
  step=2, sim=0.870: NameError: name 'sum_list' is not defined
  step=2, sim=0.884: NameError: name 'count_integer' is not defined


## 4. Iterative Agent

The generate-test-debug loop. Each condition gets the same number of total LLM calls.

In [6]:
def generate_code(prompt, error_context='', fix_context='', temperature=0.0):
    """Generate or fix code."""
    system = (
        'You are an expert Python programmer. Write a function based on the description. '
        'Return ONLY the Python code, no explanation, no markdown fences.'
    )
    
    user_parts = []
    if fix_context:
        user_parts.append(fix_context)
        user_parts.append('---\n')
    if error_context:
        user_parts.append(f'My previous attempt had this error:\n{error_context}\n')
        user_parts.append('Please fix the code.\n')
    user_parts.append(prompt)
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=temperature,
        max_tokens=1024,
    )
    code = response.choices[0].message.content.strip()
    if code.startswith('```'):
        code = code.split('\n', 1)[1] if '\n' in code else code[3:]
    if code.endswith('```'):
        code = code[:-3]
    return code.strip()


def evaluate_humaneval(prompt, code, test_code, entry_point):
    """Run HumanEval tests. Returns (passed, error_msg)."""
    full_code = prompt + code + '\n' + test_code
    try:
        exec_globals = {}
        exec(full_code, exec_globals)
        exec(f'check({entry_point})', exec_globals)
        return True, ''
    except Exception as e:
        return False, f'{type(e).__name__}: {str(e)[:200]}'


def run_single_pass(problem):
    """Single attempt, no retry."""
    code = generate_code(problem['prompt'])
    passed, _ = evaluate_humaneval(problem['prompt'], code, problem['test'], problem['entry_point'])
    return {'passed': passed, 'attempts': 1}


def run_retry_no_memory(problem, max_retries=MAX_RETRIES):
    """Retry with error feedback but no CVX memory."""
    code = generate_code(problem['prompt'])
    passed, error = evaluate_humaneval(problem['prompt'], code, problem['test'], problem['entry_point'])
    
    for attempt in range(max_retries):
        if passed:
            break
        error_ctx = f'Error: {error}\nFailing code:\n{code[:300]}'
        code = generate_code(problem['prompt'], error_context=error_ctx)
        passed, error = evaluate_humaneval(problem['prompt'], code, problem['test'], problem['entry_point'])
    
    return {'passed': passed, 'attempts': attempt + 2 if not passed else attempt + 2}


def run_retry_cvx_causal(problem, max_retries=MAX_RETRIES):
    """Retry with CVX causal error matching."""
    code = generate_code(problem['prompt'])
    passed, error = evaluate_humaneval(problem['prompt'], code, problem['test'], problem['entry_point'])
    
    for attempt in range(max_retries):
        if passed:
            break
        # CVX: search for similar errors and get fix suggestions
        suggestions = retrieve_fix_suggestions(error, code)
        fix_ctx = format_fix_context(suggestions)
        error_ctx = f'Error: {error}\nFailing code:\n{code[:300]}'
        code = generate_code(problem['prompt'], error_context=error_ctx, fix_context=fix_ctx)
        passed, error = evaluate_humaneval(problem['prompt'], code, problem['test'], problem['entry_point'])
    
    return {'passed': passed, 'attempts': attempt + 2 if not passed else attempt + 2}


# Quick test
test_problem = humaneval[5]  # pick a medium-difficulty problem
print(f'Test: {test_problem["entry_point"]}')
for name, fn in [('SinglePass', run_single_pass), ('Retry-NoMem', run_retry_no_memory), ('Retry-CVX', run_retry_cvx_causal)]:
    result = fn(test_problem)
    print(f'  {name}: passed={result["passed"]}, attempts={result["attempts"]}')

Test: intersperse


  SinglePass: passed=True, attempts=1


  Retry-NoMem: passed=True, attempts=2


  Retry-CVX: passed=True, attempts=2


## 5. Full Evaluation on HumanEval

In [7]:
conditions = {
    'SinglePass': run_single_pass,
    'Retry-NoMemory': run_retry_no_memory,
    'Retry-CVX-Causal': run_retry_cvx_causal,
}

all_results = {cond: [] for cond in conditions}
t0 = time.perf_counter()

for i, problem in enumerate(humaneval):
    for cond, fn in conditions.items():
        result = fn(problem)
        result['task_id'] = problem['task_id']
        result['entry_point'] = problem['entry_point']
        all_results[cond].append(result)
    
    if (i + 1) % 20 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (i + 1) * (len(humaneval) - i - 1)
        rates = {c: sum(r['passed'] for r in res) / len(res) for c, res in all_results.items()}
        print(f'[{i+1}/{len(humaneval)}] {rates} ({elapsed:.0f}s, ETA {eta:.0f}s)')

print(f'\nDone in {time.perf_counter() - t0:.0f}s')

print('\n=== RESULTS (full HumanEval, n=164) ===')
for cond, res in all_results.items():
    passed = sum(r['passed'] for r in res)
    total = len(res)
    mean_att = np.mean([r['attempts'] for r in res])
    print(f'{cond:>20}: {passed}/{total} = {passed/total:.1%} pass@1  (mean attempts: {mean_att:.1f})')

[20/164] {'SinglePass': 0.95, 'Retry-NoMemory': 0.95, 'Retry-CVX-Causal': 0.95} (40s, ETA 291s)


[40/164] {'SinglePass': 0.925, 'Retry-NoMemory': 0.925, 'Retry-CVX-Causal': 0.925} (84s, ETA 260s)


[60/164] {'SinglePass': 0.95, 'Retry-NoMemory': 0.95, 'Retry-CVX-Causal': 0.95} (110s, ETA 190s)


[80/164] {'SinglePass': 0.9375, 'Retry-NoMemory': 0.95, 'Retry-CVX-Causal': 0.95} (147s, ETA 154s)


[100/164] {'SinglePass': 0.91, 'Retry-NoMemory': 0.94, 'Retry-CVX-Causal': 0.93} (196s, ETA 126s)


[120/164] {'SinglePass': 0.9, 'Retry-NoMemory': 0.9333333333333333, 'Retry-CVX-Causal': 0.925} (237s, ETA 87s)


[140/164] {'SinglePass': 0.8571428571428571, 'Retry-NoMemory': 0.8928571428571429, 'Retry-CVX-Causal': 0.8857142857142857} (307s, ETA 53s)


[160/164] {'SinglePass': 0.8625, 'Retry-NoMemory': 0.89375, 'Retry-CVX-Causal': 0.8875} (348s, ETA 9s)



Done in 356s

=== RESULTS (full HumanEval, n=164) ===
          SinglePass: 141/164 = 86.0% pass@1  (mean attempts: 1.0)
      Retry-NoMemory: 146/164 = 89.0% pass@1  (mean attempts: 2.2)
    Retry-CVX-Causal: 145/164 = 88.4% pass@1  (mean attempts: 2.3)


In [8]:
# Analysis: where does CVX help?

# Problems that failed SinglePass but were rescued by retry
single_fail = {r['task_id'] for r in all_results['SinglePass'] if not r['passed']}
rescued_no_mem = {r['task_id'] for r in all_results['Retry-NoMemory'] if r['passed'] and r['task_id'] in single_fail}
rescued_cvx = {r['task_id'] for r in all_results['Retry-CVX-Causal'] if r['passed'] and r['task_id'] in single_fail}

print(f'Problems that failed SinglePass: {len(single_fail)}')
print(f'Rescued by Retry-NoMemory: {len(rescued_no_mem)}')
print(f'Rescued by Retry-CVX-Causal: {len(rescued_cvx)}')
print(f'CVX-only rescues (CVX fixed, NoMem didnt): {len(rescued_cvx - rescued_no_mem)}')
print(f'NoMem-only rescues: {len(rescued_no_mem - rescued_cvx)}')

# McNemar: Retry-CVX vs Retry-NoMemory
a_pass = np.array([r['passed'] for r in all_results['Retry-CVX-Causal']])
b_pass = np.array([r['passed'] for r in all_results['Retry-NoMemory']])
n_10 = np.sum(a_pass & ~b_pass)
n_01 = np.sum(~a_pass & b_pass)
n_11 = np.sum(a_pass & b_pass)
n_00 = np.sum(~a_pass & ~b_pass)

if n_10 + n_01 > 0:
    chi2 = (abs(n_10 - n_01) - 1) ** 2 / (n_10 + n_01)
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
else:
    chi2, p_val = 0, 1.0
sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

print(f'\n=== McNemar: Retry-CVX vs Retry-NoMemory ===')
print(f'CVX only: {n_10}, NoMem only: {n_01}, both: {n_11}, neither: {n_00}')
print(f'Net: {n_10 - n_01:+d}, chi2={chi2:.2f}, p={p_val:.4f} {sig}')

Problems that failed SinglePass: 23
Rescued by Retry-NoMemory: 5
Rescued by Retry-CVX-Causal: 4
CVX-only rescues (CVX fixed, NoMem didnt): 0
NoMem-only rescues: 1

=== McNemar: Retry-CVX vs Retry-NoMemory ===
CVX only: 0, NoMem only: 1, both: 145, neither: 18
Net: -1, chi2=0.00, p=1.0000 ns


In [9]:
import plotly.graph_objects as go

colors = {'SinglePass': '#95a5a6', 'Retry-NoMemory': '#3498db', 'Retry-CVX-Causal': '#2ecc71'}

fig = go.Figure()
for cond, res in all_results.items():
    rate = sum(r['passed'] for r in res) / len(res)
    fig.add_trace(go.Bar(
        x=[cond], y=[rate * 100],
        text=f'{rate:.1%}', textposition='outside',
        marker_color=colors.get(cond, '#333'),
    ))

fig.update_layout(
    title=f'HumanEval pass@1 with Debug Loop (n={len(humaneval)}, max {MAX_RETRIES} retries)',
    yaxis_title='pass@1 (%)', yaxis=dict(range=[0, 100]),
    height=400, showlegend=False,
)
fig.show()

# Rescue analysis: problems only CVX saved
cvx_only_tasks = rescued_cvx - rescued_no_mem
if cvx_only_tasks:
    print(f'\nProblems rescued ONLY by CVX-Causal ({len(cvx_only_tasks)}):')
    for tid in list(cvx_only_tasks)[:5]:
        prob = next(p for p in humaneval if p['task_id'] == tid)
        print(f'  {tid}: {prob["entry_point"]}')

In [10]:
# Summary
print('=== E4: ITERATIVE CODING WITH CVX DEBUG MEMORY ===')
print(f'Model: {LLM_MODEL}')
print(f'Debug corpus: {len(debug_traces)} error traces from MBPP (T=0.8)')
print(f'CVX index: {len(index)} vectors (4 steps per trace)')
print(f'Eval: HumanEval (n={len(humaneval)}), max {MAX_RETRIES} retries')
print()
for cond, res in all_results.items():
    passed = sum(r['passed'] for r in res)
    total = len(res)
    print(f'  {cond:>20}: {passed}/{total} = {passed/total:.1%}')
print()
print('CVX features: search across error embeddings, episode encoding,')
print('temporal continuation (error→fix), iterative agent loop')
print()
print('This is the CODE GENERATION equivalent of E3:')
print('the query evolves at each retry based on the ACTUAL error,')
print('not the static problem description.')

=== E4: ITERATIVE CODING WITH CVX DEBUG MEMORY ===
Model: qwen2.5-coder:7b-instruct
Debug corpus: 353 error traces from MBPP (T=0.8)
CVX index: 1412 vectors (4 steps per trace)
Eval: HumanEval (n=164), max 3 retries

            SinglePass: 141/164 = 86.0%
        Retry-NoMemory: 146/164 = 89.0%
      Retry-CVX-Causal: 145/164 = 88.4%

CVX features: search across error embeddings, episode encoding,
temporal continuation (error→fix), iterative agent loop

This is the CODE GENERATION equivalent of E3:
the query evolves at each retry based on the ACTUAL error,
not the static problem description.
